<a href="https://colab.research.google.com/github/satyagalla/deepfake/blob/main/debug.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Counterfactual Shortcut Probe (debugging)

Interactive companion to `model/counterfactual_probe.py` -- same functions, run cell-by-cell so
templates and individual predictions can be inspected before trusting the summary numbers.

Tests the hypothesis in `docs/investigations/2026-07-26-upscale-artifact.md`: keep an image's true
`rgb` content, swap its `fft_mag`/`srm_residual` for another class's averaged template, and check
whether the trained model's prediction follows content or follows the swapped artifact channels.

## Setup

In [1]:
import os
if os.path.isdir("deepfake"):
    %cd deepfake
    !git pull
else:
    !git clone https://github.com/satyagalla/deepfake.git
    %cd deepfake

/content/deepfake
Already up to date.


In [2]:
%matplotlib inline

In [3]:
!pip install -q facenet-pytorch timm kaggle datasets

import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')


CUDA available: True
GPU: NVIDIA L4


In [4]:
# Mount Drive -- config.py's CHECKPOINT_DIR/DATASET_DIR/EVAL_DIR all resolve under
# DEEPFAKE_DATA_ROOT, so this must be set before any `config`/`model.*` import.
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ['DEEPFAKE_DATA_ROOT'] = '/content/drive/MyDrive/deepfake'


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import matplotlib.pyplot as plt

from config import CHECKPOINT_DIR, CLASSES, DEVICE
from model.dataset import ForgeryDataset
from model.eval import load_model
from model.counterfactual_probe import (
    _class_indices,
    extract_templates,
    run_probe,
    summarize,
    print_summary,
    swap_target_spread,
    print_swap_target_spread,
    save_probe_results,
    _predict,
)

print("device:", DEVICE)

[transformers] Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


device: cuda


## Load model + dataset

In [6]:
checkpoint_path = CHECKPOINT_DIR / "best_model.pt"  # edit if your checkpoint has a different name

model = load_model(str(checkpoint_path))
dataset = ForgeryDataset("val")
indices_by_class = _class_indices(dataset)

{cls: len(idxs) for cls, idxs in indices_by_class.items()}


model.safetensors: reconstructing file:   0%|          |  0.00B / 77.9MB            

model.safetensors: downloading bytes:           |  0.00B            

{'real': 291, 'edited': 162, 'deepfake': 313}

## Build per-class templates

Small counts here on purpose for a fast debug loop -- bump `TEMPLATE_N` once this looks right.

In [7]:
TEMPLATE_N = 100  # samples per class averaged into that class's fft/srm template

templates = extract_templates(dataset, indices_by_class, TEMPLATE_N)
{cls: (t["fft_mag"].shape, t["srm_residual"].shape) for cls, t in templates.items()}


{'real': (torch.Size([1, 380, 380]), torch.Size([9, 380, 380])),
 'edited': (torch.Size([1, 380, 380]), torch.Size([9, 380, 380])),
 'deepfake': (torch.Size([1, 380, 380]), torch.Size([9, 380, 380]))}

## Sanity check: do the templates visually look distinct?

`fft_mag` is single-channel and directly viewable. `srm_residual` is 9-channel (3 SRM kernels x
3 RGB channels) -- shown here is channel 0 only, just to eyeball whether the averaging produced a
sensible (non-degenerate) residual map rather than something that collapsed to near-zero/noise.

In [8]:
import numpy as np

fig, axes = plt.subplots(2, len(CLASSES), figsize=(4 * len(CLASSES), 8))
for col, cls in enumerate(CLASSES):
    axes[0, col].imshow(templates[cls]["fft_mag"][0], cmap="viridis")
    axes[0, col].set_title(f"{cls}: fft_mag template")
    axes[0, col].axis("off")

    resid = templates[cls]["srm_residual"][0].numpy()
    lim = np.percentile(np.abs(resid), 99)
    axes[1, col].imshow(resid, cmap="gray", vmin=-lim, vmax=lim)
    axes[1, col].set_title(f"{cls}: srm_residual[0]")
    axes[1, col].axis("off")

fig.tight_layout()

In [9]:
import itertools

pairs = list(itertools.combinations(CLASSES, 2))
fig, axes = plt.subplots(2, len(pairs), figsize=(4 * len(pairs), 8))
for col, (a, b) in enumerate(pairs):
    fft_diff = (templates[a]["fft_mag"][0] - templates[b]["fft_mag"][0]).numpy()
    srm_diff = (templates[a]["srm_residual"][0] - templates[b]["srm_residual"][0]).numpy()

    fft_lim = np.percentile(np.abs(fft_diff), 99)
    srm_lim = np.percentile(np.abs(srm_diff), 99)

    axes[0, col].imshow(fft_diff, cmap="RdBu_r", vmin=-fft_lim, vmax=fft_lim)
    axes[0, col].set_title(f"fft_mag: {a} - {b}")
    axes[0, col].axis("off")

    axes[1, col].imshow(srm_diff, cmap="RdBu_r", vmin=-srm_lim, vmax=srm_lim)
    axes[1, col].set_title(f"srm_residual[0]: {a} - {b}")
    axes[1, col].axis("off")

fig.tight_layout()

## Run the probe on a few held-out samples

`PROBE_N` samples per class, disjoint from the `TEMPLATE_N` used above (`run_probe` slices
`idxs[TEMPLATE_N : TEMPLATE_N + PROBE_N]` internally) -- baseline (own fft/srm) is printed
alongside every swap so the shift can be read per image.

In [10]:
PROBE_N = 30

results = run_probe(model, dataset, indices_by_class, templates, PROBE_N, TEMPLATE_N)
len(results)



[real] /content/drive/MyDrive/deepfake/dataset/val/real/real_03746_00100.jpg
  baseline (own fft/srm) : pred=     real  probs=real=0.986, edited=0.002, deepfake=0.012
  swap->real     fft  [same-class (control)]: pred=     real  probs=real=0.986, edited=0.002, deepfake=0.012
  swap->real     srm  [same-class (control)]: pred=     real  probs=real=0.995, edited=0.000, deepfake=0.005
  swap->real     both [same-class (control)]: pred=     real  probs=real=0.995, edited=0.000, deepfake=0.004
  swap->edited   fft  [cross-class         ]: pred=     real  probs=real=0.986, edited=0.002, deepfake=0.012
  swap->edited   srm  [cross-class         ]: pred=     real  probs=real=0.994, edited=0.000, deepfake=0.005
  swap->edited   both [cross-class         ]: pred=     real  probs=real=0.995, edited=0.001, deepfake=0.005
  swap->deepfake fft  [cross-class         ]: pred=     real  probs=real=0.986, edited=0.002, deepfake=0.012
  swap->deepfake srm  [cross-class         ]: pred=     real  probs=r

90

## Inspect one sample in detail

Re-run with a different index into `results` to look at other probed images.

In [11]:
r = results[0]
print("path:", r["path"])
print("true_class:", r["true_class"])
print("baseline:", r["baseline"])
print()
for swap_cls, variants in r["swaps"].items():
    tag = "same-class (control)" if swap_cls == r["true_class"] else "cross-class"
    print(f"-- swap -> {swap_cls} [{tag}] --")
    for variant, pred in variants.items():
        print(f"  {variant:>4s}: {pred}")


path: /content/drive/MyDrive/deepfake/dataset/val/real/real_03746_00100.jpg
true_class: real
baseline: {'pred': 'real', 'probs': {'real': 0.9860527515411377, 'edited': 0.001570167951285839, 'deepfake': 0.01237706933170557}, 'gate': {'spatial': 0.2877879738807678, 'spectral': 0.36771345138549805, 'noise_residual': 0.34449857473373413}}

-- swap -> real [same-class (control)] --
   fft: {'pred': 'real', 'probs': {'real': 0.9858877062797546, 'edited': 0.0017898563528433442, 'deepfake': 0.0123224388808012}, 'gate': {'spatial': 0.2868732213973999, 'spectral': 0.36658450961112976, 'noise_residual': 0.34654226899147034}}
   srm: {'pred': 'real', 'probs': {'real': 0.9951499104499817, 'edited': 0.0003472610842436552, 'deepfake': 0.004502890631556511}, 'gate': {'spatial': 0.2685627043247223, 'spectral': 0.3779253661632538, 'noise_residual': 0.3535119891166687}}
  both: {'pred': 'real', 'probs': {'real': 0.9951204657554626, 'edited': 0.00039590231608599424, 'deepfake': 0.004483633209019899}, 'gat

## Summary

`cross_class - same_class` is the clean signal: `same_class` alone is just the noise floor from
using an averaged template instead of the image's own exact fft/srm values.

In [12]:
summary = summarize(results)
print_summary(summary)



=== Summary: mean delta in P(swap_class) vs baseline ===
variant   same_class (noise floor)  cross_class (shortcut signal)   cross - same
   fft                     0.0042                        -0.0018        -0.0059
   srm                    -0.0381                         0.0220         0.0601
  both                    -0.0287                         0.0176         0.0462


## Does swap-*target* identity actually matter?

`cross - same` above only shows a swap moved the prediction -- it can't tell a targeted
per-class shortcut apart from "any averaged template moves things the same way regardless
of source class" (a generic averaging artifact, not class-specific). This compares each
image's 3 swap-target predictions against *each other*: a small spread means swap identity
barely matters.

In [13]:
from model.counterfactual_probe import swap_target_spread, print_swap_target_spread

spread = swap_target_spread(results)
print_swap_target_spread(spread)


=== Swap-target spread: does *which* class's template was used matter? ===
variant  mean spread   max spread
   fft       0.0016       0.0155
   srm       0.0041       0.0396
  both       0.0044       0.0596


## Save (optional)

Only run once satisfied with `TEMPLATE_N`/`PROBE_N` -- writes to `EVAL_DIR`.

In [14]:
save_probe_results(results, summary, spread)


Saved probe results -> /content/drive/MyDrive/deepfake/eval/counterfactual_probe_20260728T012003Z.json


PosixPath('/content/drive/MyDrive/deepfake/eval/counterfactual_probe_20260728T012003Z.json')